In [ ]:
import fiftyone as fo

import fiftyone.zoo as foz 

dataset = foz.load_zoo_dataset("quickstart-video", overwrite=True, persistent=True)

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

# Register the local directory as a remote source zoo
foz.register_zoo_model_source(
    "https://github.com/harpreetsahota204/vggt_omega",
    overwrite=True,
)

# Load the model
model = foz.load_zoo_model(
    "facebook/VGGT-Omega-1B-512",
    # install_requirements=False,  # set True on first run to install deps
    confidence_threshold=50.0,
    video_sample_fps=20.0,           # target fps; auto-reduced for long videos
    max_frames=160,                  # hard cap — ~10GB on A100 at default resolution
    preprocessing_mode="balanced",  # "balanced" or "max_size" (lower memory)
    image_resolution=512,           # fixed for this checkpoint, do not change
)


In [ ]:
# Required: populate sample.metadata so the model can read frame_rate,
# total_frame_count, and duration without reopening each video file.
dataset.compute_metadata()

# "depth_map" becomes the per-frame Heatmap field name.
# "scene_3d" (fo3d path) is stored as a sample-level side-effect inside predict().
dataset.apply_model(model, "depth_map", skip_failures=False)

In [ ]:
import fiftyone as fo
from pathlib import Path


def build_vggt_omega_grouped_dataset(source_dataset, name="vggt_omega_results", overwrite=True):
    """Build a grouped dataset from VGGT-Omega apply_model output.

    Slices:
      - "video"  : original video + per-frame depth Heatmaps
      - "threed" : merged .fo3d 3D scene
    """
    grouped = fo.Dataset(name, overwrite=overwrite)
    grouped.add_group_field("group", default="video")

    samples = []

    for source_sample in source_dataset.iter_samples(progress=True):
        path = Path(source_sample.filepath)
        base_dir = path.parent
        stem = path.stem
        group = fo.Group()

        # Video slice — attach depth PNGs as Heatmaps.
        # Files are named {stem}_frame_{i:06d}_depth.png (0-based i),
        # mapped to 1-based FiftyOne frame numbers.
        video_sample = fo.Sample(filepath=str(path), group=group.element("video"))
        for i, depth_png in enumerate(sorted(base_dir.glob(f"{stem}_frame_*_depth.png"))):
            video_sample.frames[i + 1]["depth_map"] = fo.Heatmap(map_path=str(depth_png))
        samples.append(video_sample)

        # 3D slice
        fo3d = base_dir / f"{stem}_scene.fo3d"
        if fo3d.exists():
            samples.append(fo.Sample(filepath=str(fo3d), group=group.element("threed")))

    grouped.add_samples(samples)
    return grouped


grouped_dataset = build_vggt_omega_grouped_dataset(dataset)
print(grouped_dataset)

In [ ]:
session = fo.launch_app(grouped_dataset, auto=False)
session.url